# Predict with CMV - OpenCV und Catboost-Models
Conda Env: scikit-learn
last edit: 15.08.2026

In [ ]:
import numpy as np
import pandas as pd
import cv2
import os
from osgeo import gdal
gdal.UseExceptions()
from datetime import datetime, timedelta
import datetime
import shlex
import catboost
from catboost import *
import sklearn.metrics as metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
import rasterio
import plotly.express as px
import plotly.graph_objects as go

### set variables

In [2]:
# dictionary with prediction scenarios for testing the prediction model
PREDICTION_SCENARIOS = [
    {"timestamp": datetime.datetime.strptime("29.01.2018 07:35:11", "%d.%m.%Y %H:%M:%S"),"description": "Clear winter morning with shadows from the mountains"},
    {"timestamp": datetime.datetime.strptime("22.03.2018 14:33:53", "%d.%m.%Y %H:%M:%S"),"description": "Cloudy evening"},
    {"timestamp": datetime.datetime.strptime("11.08.2018 07:33:53", "%d.%m.%Y %H:%M:%S"),"description": "Cloudy morning"},
    {"timestamp": datetime.datetime.strptime("12.08.2018 12:13:53", "%d.%m.%Y %H:%M:%S"),"description": "Sunny day"},
    # {"timestamp": datetime.datetime.strptime("14.08.2018 12:37:23", "%d.%m.%Y %H:%M:%S"),"description": "Rainstorm from north west (half hour later)"}, # commented out because the time overlapping with outer scenario
    {"timestamp": datetime.datetime.strptime("14.08.2018 12:13:53", "%d.%m.%Y %H:%M:%S"),"description": "Rainstorm from north west"},
    {"timestamp": datetime.datetime.strptime("14.11.2018 09:07:51", "%d.%m.%Y %H:%M:%S"),"description": "Autumn day with fog in lower regions"},
    {"timestamp": datetime.datetime.strptime("16.12.2018 11:02:21", "%d.%m.%Y %H:%M:%S"),"description": "Winter day with snow and clouds"},
]

# set path variables
eumetsat_path = "C:\\Users\\Andreas\\Documents\\UNIGIS\\2017\\Master-Thesis\\Daten\\Satellite\\EUMETSAT"
eumetsat_geotiff_path = eumetsat_path + "\\Result_Timestamped\\GeoTIFF\\Extended_Clip" # path to Extended clipped EUMETSAT-GeoTIFF images
eumetsat_forecast_output_path = eumetsat_path + "\\Forecast" # patch to save forecast geotiff
result_path = r'.\results' # path for result diagrams/CSVs

# set list with all SEVIRI bands and correct order
# source: https://eumetsat.int/0-degree-service
all_bands = ['HRV','VIS006','VIS008','IR_016','IR_039','WV_062','WV_073','IR_087','IR_097','IR_108','IR_120','IR_134']
#all_bands = ["HRV"]

# set forecast range in step in minutes
forecast_range = 180
forecast_step = 15

### map feature sets to their trained CBM model files

In [3]:
# set FEATURE_SETS dict as used during training of the CatBoost models
GIS_TIME = ['HEIGHT', 'PV_SUMME_INSTALLIERT','SOLAR_RADIATION_GLOBALRAD', 'SOLAR_RADIATION_DIRECTRAD', 'SOLAR_RADIATION_DIFFUSERAD',
            'HOURDEZ', 'DAYYEAR', 'SIN_HOUR','COS_HOUR', 'SIN_DAY', 'COS_DAY']
WMS4 = ['VIS006', 'IR_039', 'WV_062', 'IR_108']
ALL12 = ['VIS006', 'VIS008', 'IR_016', 'IR_039', 'WV_062', 'WV_073','IR_087', 'IR_097', 'IR_108', 'IR_120', 'IR_134', 'HRV']
SENSOR = ['GLOBAL_RADIATION_SENSOR_VALUE', 'GLOBAL_RADIATION_SENSOR_TEMPERATURE']

# feature set with Name, features and model file per feature set
FEATURE_SETS = {
    'FS1_BASELINE'       : {'features': GIS_TIME,               'cbm': './Models/model_FS1_BASELINE.cbm'},
    'FS2_NOSENSOR'       : {'features': GIS_TIME + WMS4,        'cbm': './Models/model_FS2_NOSENSOR.cbm'},
    'FS3_ALLSEVIRI'      : {'features': GIS_TIME + ALL12,       'cbm': './Models/model_FS3_ALLSEVIRI.cbm'},
    'FS4_WITHSENSOR'     : {'features': GIS_TIME + WMS4 + SENSOR, 'cbm': './Models/model_FS4_WITHSENSOR.cbm'},
    'FS5_SENSORONLY'     : {'features': GIS_TIME + SENSOR,      'cbm': './Models/model_FS5_SENSORONLY.cbm'},
    'FS6_SEVIRI_SENSOR'  : {'features': GIS_TIME + ALL12 + SENSOR, 'cbm': './Models/model_FS6_SEVIRI_SENSOR.cbm'},
}

# load all models once into a dictonary for later use
loaded_models = {}
for fs_name, fs_conf in FEATURE_SETS.items():
    m = CatBoostRegressor()
    m.load_model(fs_conf['cbm'])
    loaded_models[fs_name] = m
    print("loaded model for", fs_name, fs_conf['cbm'])

loaded model for FS1_BASELINE ./Models/model_FS1_BASELINE.cbm
loaded model for FS2_NOSENSOR ./Models/model_FS2_NOSENSOR.cbm
loaded model for FS3_ALLSEVIRI ./Models/model_FS3_ALLSEVIRI.cbm
loaded model for FS4_WITHSENSOR ./Models/model_FS4_WITHSENSOR.cbm
loaded model for FS5_SENSORONLY ./Models/model_FS5_SENSORONLY.cbm
loaded model for FS6_SEVIRI_SENSOR ./Models/model_FS6_SEVIRI_SENSOR.cbm


### set date and time for last and next to last Image  

In [4]:
# Function: to round actual/predict time to Quarter
def compute_rounded_times(date_predict):
    # date_last_image_rounded = date_predict - (date_predict - date_predict.min) % timedelta(minutes=15) # not safe method
    date_last_image_rounded = date_predict - timedelta(minutes=date_predict.minute % 15,seconds=date_predict.second,microseconds=date_predict.microsecond)
    date_last_image=date_last_image_rounded.strftime("%Y-%m-%d %H_%M_%S")
    print("actual/predict datetime:             {}".format(date_predict)) # pring date an time
    print("rounded datetime last image:         {}".format(date_last_image))  # printed in default formatting
    # substract 15min from date for next to last image
    date_next_to_last_rounded = date_last_image_rounded - datetime.timedelta(minutes=15)
    date_next_to_last_image=date_next_to_last_rounded.strftime("%Y-%m-%d %H_%M_%S")
    print("rounded datetime next to last image: {}".format(date_next_to_last_image))  # printed in default formatting
    return date_last_image_rounded, date_last_image, date_next_to_last_image

# for testing the function
# date_predict = datetime.datetime.strptime("29.01.2018 07:35:11", "%d.%m.%Y %H:%M:%S")
# compute_rounded_times(date_predict)

In [5]:
# Function: load all images and bands from GeoTIFF into a nested dictionary
def load_images_and_bands(date_last_image, date_next_to_last_image, eumetsat_geotiff_path,all_bands):
    images=["next_to_last_image","last_image"]

    # create empty nested dictionary for images and bands
    images_data = {}
    for image in images:
        images_data[image] = {}

    # loop for next to last and last images
    for image in images:
        if image == "next_to_last_image":
            date_processing = date_next_to_last_image
        else:
            date_processing = date_last_image

        datasets = ["IR_VIS_WR","HRV"]
                
        # loop trough all datasets
        for dataset in datasets:
            # set bands for each dataset
            if dataset == "HRV":
                bands = ["HRV"]
            else:
                bands = all_bands[1:] # exclude HRV from all_bands list for IR_VIS_WR dataset
                
            # set starting band number for iteration
            band_nr = 1

            # loop through all bands of the given dataset
            for band in bands:
                
                # set path and filename dynamicly
                geotiff_filename = os.path.join(eumetsat_geotiff_path, date_processing + "_{}.tif".format(dataset))        

                # print for debugging
                # print("### processing file:",geotiff_filename,band,band_nr)
                
                # read geotiff-images
                image_processing = gdal.Open(geotiff_filename)
                
                # create numpy arrays with dynamic names from variable image and band e.g. "next_to_last_IR_039"
                dynamic_array = np.array(image_processing.GetRasterBand(band_nr).ReadAsArray().astype(np.float32) )
                dynamic_array_name = image + "_"+ band
                # globals()[dynamic_array_name] = dynamic_array
                images_data[image][band] = dynamic_array

                # increase the band number
                band_nr = band_nr + 1

    # get gdal-parameters from last processing image for writing results as geotiff
    gdal_params = {
        "data_type" : image_processing.GetRasterBand(1).DataType,
        "geotransform" : image_processing.GetGeoTransform(),
        "spatialreference" : image_processing.GetProjection(),
        "ncol" : image_processing.RasterXSize,
        "nrow" : image_processing.RasterYSize,
        "nband" : 1
    }

    return images_data, gdal_params

# for testing the function
# images_data, gdal_params = load_images_and_bands(
#     date_last_image, date_next_to_last_image, eumetsat_geotiff_path
# )

# print(array_next_to_last_vis006.shape, array_next_to_last_vis006.dtype)

### Detect motion flow from HRV-Files and predict on all bands

In [6]:
# Function: export_geotiff for export result as single-band GeoTIFF-Raster
def export_geotiff(path, file, band, ncol, nrow, nband, data_type, geotransform, spatialreference, image):
    # create output folder for geotiff
    if not os.path.exists(path + "\\" + band):
        os.makedirs(path + "\\" + band)

    # set geotiff filename
    output_geotiff_file = os.path.join(path + "\\" + band, file + ".tif")
    
    # create geotiff file of forecast
    # Source: https://borealperspectives.org/2014/01/16/data-type-mapping-when-using-pythongdal-to-write-numpy-arrays-to-geotiff/
    driver = gdal.GetDriverByName("GTiff")
    # Source: https://kokoalberti.com/articles/geotiff-compression-optimization-guide/
    #dst_dataset = driver.Create(output_geotiff_file, ncol, nrow, nband, data_type,  [ 'COMPRESS=ZSTD', 'PREDICTOR=3', 'TILED=YES', 'NUM_THREADS=ALL_CPUS' ])
    dst_dataset = driver.Create(output_geotiff_file, ncol, nrow, nband, data_type,  [ 'COMPRESS=PACKBITS', 'TILED=YES', 'NUM_THREADS=ALL_CPUS' ])
    dst_dataset.SetGeoTransform(geotransform)
    dst_dataset.SetProjection(spatialreference)
    dst_dataset.GetRasterBand(1).SetDescription(band)
    dst_dataset.GetRasterBand(1).WriteArray(image)
    dst_dataset = None

In [7]:
# Function: print array min/max/median/mean for debugging
def print_array(name, array):
    print("  --{} min:    {}".format(name,str(np.min(array))))
    print("    {} max:    {}".format(name,str(np.max(array))))
    print("    {} median: {}".format(name,str(np.median(array))))
    print("    {} mean:   {}".format(name,str(np.mean(array))))

In [8]:
# Function: array2raster2 for export a array as Single Band-Geotiff Raster
def array2raster2(path, fname, matriz, geot, proj):
    # create output folder for geotiff
    if not os.path.exists(path):
        os.makedirs(path)

    # Source: https://gis.stackexchange.com/questions/189942/writing-3-channels-to-8-bit-tif-in-python-using-gdal
    drv = gdal.GetDriverByName("GTiff")
    dst_ds = drv.Create(os.path.join(path + "\\", fname), matriz.shape[1], matriz.shape[0], 3, gdal.GDT_Byte)
    dst_ds.SetGeoTransform(geot)
    dst_ds.SetProjection(proj)
    dst_ds.GetRasterBand(3).WriteArray(matriz[:, :, 0])  
    dst_ds.GetRasterBand(2).WriteArray(matriz[:, :, 1])  
    dst_ds.GetRasterBand(1).WriteArray(matriz[:, :, 2])
    dst_ds.FlushCache()
    dst_ds=None

In [9]:
# Function: processOptical Flow + Export results as GeoTIFF-Raster
def run_optical_flow_and_export(images_data, gdal_params, date_last_image, forecast_range, forecast_step,eumetsat_forecast_output_path,all_bands):

    # get gdal parameters for writing results as geotiff
    data_type = gdal_params["data_type"]
    geotransform = gdal_params["geotransform"]
    spatialreference = gdal_params["spatialreference"]
    ncol = gdal_params["ncol"]
    nrow = gdal_params["nrow"]
    nband = gdal_params["nband"]

    ### create and normalize arrays to 8 bit (0-255) for CMV with OpenCV

    # create empty nested dictionary for scaled data
    scaled_data = {}

    for label in ["next_to_last_image", "last_image"]:
        scaled_data[label] = {}

    # loop trough all bands
    for band in all_bands:
        print(f"### processing values: next_to_last_{band} last_image_{band}")
        next_to_last_image_processing_array = images_data["next_to_last_image"][band]
        last_image_processing_array = images_data["last_image"][band]

        # concatenate arrays of each band for getting min/max values for scaling
        concatenated_array = np.concatenate((next_to_last_image_processing_array, last_image_processing_array), axis=0)

        # normalize arrays to concatenated max scale 255 within 0 and 255
        next_to_last_image_scaled = np.uint8((next_to_last_image_processing_array) / (np.max(concatenated_array)) * 255)
        last_image_scaled = np.uint8((last_image_processing_array) / (np.max(concatenated_array)) * 255)

        # print min/max values for scale debugging
        # print_array(f"next_to_last_{band}_processing_array", next_to_last_image_processing_array)
        # print_array(f"last_image_{band}_processing_array", last_image_processing_array)
        # print_array(f"{band}_concatenated_array", concatenated_array)
        # print_array(f"next_to_last_{band}_scaled", next_to_last_image_scaled)
        # print_array(f"last_image_{band}_scaled", last_image_scaled)

        # Ergebnisse im Dictionary statt in globals() ablegen
        scaled_data["next_to_last_image"][band] = next_to_last_image_scaled
        scaled_data["last_image"][band] = last_image_scaled

        # export the two base images (next to last and last) as geotiff raster
        export_geotiff(eumetsat_forecast_output_path, date_last_image + " -15min_" + band, band, ncol, nrow, nband, data_type, geotransform, spatialreference,next_to_last_image_processing_array)
        export_geotiff(eumetsat_forecast_output_path, date_last_image + " +0min_" + band, band,ncol, nrow, nband, data_type, geotransform, spatialreference,last_image_processing_array)

    return scaled_data

In [10]:
# Function: visualize Flow-Motion from HRV-Band with arrows
def draw_flow(img, flow, step=16):
    h, w = img.shape[:2]
    y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)
    fx, fy = flow[y,x].T*5
    lines = np.vstack([x, y, x+fx, y+fy]).T.reshape(-1, 2, 2)
    lines = np.int32(lines + 0.5)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    #cv2.polylines(vis, lines, 0, (0, 0, 255))
    for (x1, y1), (_x2, _y2) in lines:
        # cv2.circle(vis, (x1, y1), 1, (0, 0, 255), -1)
        cv2.arrowedLine(vis, (x1, y1), (_x2, _y2), (0, 255, 0), (1), 1, 0, 0.5 )
    return vis

In [11]:
# Function: compute Optical Flow forecast for all steps and export as GeoTIFF
def compute_optical_flow_forecast(images_data,scaled_data,gdal_params,date_last_image,forecast_range,forecast_step,eumetsat_forecast_output_path,all_bands):

    # get gdal parameters for writing results as geotiff
    data_type = gdal_params["data_type"]
    geotransform = gdal_params["geotransform"]
    spatialreference = gdal_params["spatialreference"]
    ncol = gdal_params["ncol"]
    nrow = gdal_params["nrow"]
    nband = gdal_params["nband"]

    # create a nested dictionary for scaled and array data
    state = {
        "scaled": {"next_to_last_image": dict(scaled_data["next_to_last_image"]),"last_image": dict(scaled_data["last_image"]),},
        "array": {"next_to_last_image": dict(images_data["next_to_last_image"]),"last_image": dict(images_data["last_image"]), }
    }

    forecast_results = {}  # create empty dictionary for forecast results

    # Optical-Flow-Algorithmus (Parameter nach Urbich et al. 2018)
    # Parameters from "A Novel Approach for the Short-Term Forecast of the Effective Cloud Albedo" by Isabel Urbich et al. 2018
            # https://www.mdpi.com/2072-4292/10/6/955/htm#table_body_display_remotesensing-10-00955-t002
            #optical_flow = cv2.optflow.DualTVL1OpticalFlow_create(0.1, 0.03, 0.3, 3, 3, 0.01, 10, 2, 0.5, 0.1, 5, 0 ) # Result predicition r2=0.27
    
            # Documentation:  DualTVL1OpticalFlow_create([, tau [, lambda[, theta[, nscales[, warps[, epsilon[, innnerIterations[, outerIterations[, scaleStep[, gamma[, medianFiltering[, useInitialFlow]]]]]]]]]]]])
            # Default Values: DualTVL1OpticalFlow_create([, 0.25[, 0.15  [, 0.3  [, 5      [, 5    [, 0.01   [, 30              [, 10             [, 0.8      [, 0.0  [, 5              [, false         ]]]]]]]]]]]])
            # recom. Values:  DualTVL1OpticalFlow_create([, 0.1 [, 0.03  [, 0.3  [, 3      [, 3    [, 0.01   [, 10              [, 2              [, 0.5      [, 0.1  [, 5              [, false         ]]]]]]]]]]]])
    
    opticalflow = cv2.optflow.DualTVL1OpticalFlow_create(0.1, 0.003, 0.3, 6, 6, 0.005, 30, 2, 0.5, 0.2, 5, 0)

    # Use Hue, Saturation, Value colour model q
    hsv = np.zeros([nrow,ncol,3], dtype=np.uint8)
    hsv[..., 1] = 255

    # Set counter
    counter = forecast_step

    # loop trough forecast range with stepsize in minutes - step=15min and range=180min
    while counter <= forecast_range:
        # set forecast name variable
        forecast_name = date_last_image + " +" + str(counter) + "min"
        print(f"\n### processing Flow {forecast_name}")

        # calculate optical flow between next to last and last image as array from HRV because it has the highest resolution
        next_to_last_hrv_scaled = state["scaled"]["next_to_last_image"]["HRV"]
        last_hrv_scaled = state["scaled"]["last_image"]["HRV"]

        flow = opticalflow.calc(next_to_last_hrv_scaled, last_hrv_scaled, None)
        print_array("flow", flow)
        
        optical_flow = cv2.optflow.DualTVL1OpticalFlow_create(0.1, 0.003, 0.3, 6, 6, 0.005, 30, 2, 0.5, 0.2, 5, 0 )

        # Optical Flow with Farnebäck method (not used in final version)
        # calculate optical flow between next to last and last image as array with farnebäck method
        # Farnebäck, G. (2003, June). Two-frame motion estimation based on polynomial expansion. In Scandinavian conference on Image analysis (pp. 363-370). Springer, Berlin, Heidelberg.
        # flow = cv2.calcOpticalFlowFarneback(next_to_last_image_HRV_scaled, last_image_HRV_scaled, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # transform Flow-Field for cv2.remap() backward mapping
        h, w = flow.shape[:2]
        flow_transformed = -flow
        flow_transformed[..., 0] += np.arange(w)
        flow_transformed[..., 1] += np.arange(h)[:, np.newaxis]

        forecast_results[counter] = {}
      
        # print flow values for debugging
        print("### processing Flow:",forecast_name)
        print_array("flow",flow)

        # flow.fill(1)
        # print(flow.shape)
        # print("Flow : : 0")
        # print(flow[:][:][0])
        # print("Flow : : 1")
        # print(flow[:][:][1])

        # convert flow array in color image to visualize the flow
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        hsv[..., 0] = ang * 180 / np.pi / 2
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
        colored_flow = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)   
    
        # transform flow array for generating a new frame
        # Snippet from https://github.com/opencv/opencv/issues/11068
        h, w = flow.shape[:2]
        flow_transformed = -flow
        flow_transformed[:,:,0] += np.arange(w)
        flow_transformed[:,:,1] += np.arange(h)[:,np.newaxis]

        forecast_results[counter] = {}
        
        # use calculated flow to loop trough all bands to forecast
        for band in all_bands:
            last_scaled = state["scaled"]["last_image"][band]
            last_array = state["array"]["last_image"][band]

            # Forecast the next frame using the flow field and remap the last image
            forecast_scaled = cv2.remap(last_scaled, flow_transformed, None,interpolation=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT, borderValue=255)

            # print min/max values for scale debugging
            if band =="HRV":
                print_array("forecast_scaled",forecast_scaled)

                # # show colored flow in window
                # window_name_colored_flow = "Colored flow " + forecast_name
                # cv2.namedWindow(window_name_colored_flow, cv2.WINDOW_NORMAL)
                # cv2.resizeWindow(window_name_colored_flow, 900, 900)
                # cv2.imshow(window_name_colored_flow, colored_flow)
                
                print("### processing flow arrow:")
                flow_arrow = draw_flow(forecast_scaled, flow, 25)
                
                # # show flow as arrows with last HRV-image in windows
                # window_name_flow_as_arrow = "Flow as Arrow " + forecast_name
                # cv2.namedWindow(window_name_flow_as_arrow, cv2.WINDOW_NORMAL)
                # cv2.resizeWindow(window_name_flow_as_arrow, 900, 900)
                # cv2.imshow(window_name_flow_as_arrow,flow_arrow)

                # export forecast flow as geotiff file
                array2raster2(eumetsat_forecast_output_path + "\\FLOW", forecast_name + "_FLOW.tif", colored_flow, geotransform, spatialreference)
                
                # export forecast HRV with Arrows as geotiff file
                array2raster2(eumetsat_forecast_output_path + "\\HRV_ARROWS", forecast_name + "_HRV_ARROWS.tif", flow_arrow, geotransform, spatialreference)

            # scale value from 255 (greyscale) to max value from last image
            forecast_array = (forecast_scaled.astype(np.float32) / 255) * np.max(last_array.astype(float))
            
            if band == "HRV":
                print_array("forecast_scaled", forecast_scaled)
                print_array("forecast_array", forecast_array)

            # export forecast as geotiff file
            export_geotiff(eumetsat_forecast_output_path,forecast_name + "_" + band,band,ncol,nrow,nband,data_type,geotransform,spatialreference,forecast_array)

            # get pixel value for debuging https://gis.stackexchange.com/questions/118397/storing-result-from-gdallocationinfo-as-variable-in-python
            # lat=47.487297
            # lon=9.737721
            # result = os.popen('C:/Progra~1/QGIS3~1.10/bin/gdallocationinfo -valonly -wgs84 "' + os.path.join(eumetsat_forecast_output_path + "\\" + band, forecast_band_name) + '.tif" ' + str(lon) + ' ' + str(lat)).read()
            # print("    Prognose: ", forecast_band_name, result)

            # Switch - Make last image to next to last frame for each band (next_to_last <- last, last <- forecast)
            state["scaled"]["next_to_last_image"][band] = last_scaled.copy()
            state["scaled"]["last_image"][band] = forecast_scaled.copy()
            state["array"]["next_to_last_image"][band] = last_array.copy()
            state["array"]["last_image"][band] = forecast_array.copy()

        # Increment counter with forecast_step before next frame
        counter += forecast_step
        cv2.waitKey(0)

    cv2.destroyAllWindows()

In [12]:
# loop through all prediction scenarios

all_scenario_results = []

for scenario in PREDICTION_SCENARIOS:
    date_predict = scenario["timestamp"]
    print(f"\n#### Scenario: {scenario['description']} ({date_predict})")

    date_last_image_rounded, date_last_image, date_next_to_last_image = compute_rounded_times(date_predict)

    images_data, gdal_params = load_images_and_bands(date_last_image,date_next_to_last_image,eumetsat_geotiff_path,all_bands)

    scaled_data = run_optical_flow_and_export(images_data,gdal_params,date_last_image,forecast_range,forecast_step,eumetsat_forecast_output_path,all_bands)

    forecast_results = compute_optical_flow_forecast(images_data,scaled_data,gdal_params,date_last_image,forecast_range,forecast_step,eumetsat_forecast_output_path,all_bands)

    all_scenario_results.append({
        "scenario": scenario["description"],
        "timestamp": date_predict,
        "forecast_results": forecast_results
    })


#### Scenario: Clear winter morning with shadows from the mountains (2018-01-29 07:35:11)
actual/predict datetime:             2018-01-29 07:35:11
rounded datetime last image:         2018-01-29 07_30_00
rounded datetime next to last image: 2018-01-29 07_15_00
### processing values: next_to_last_HRV last_image_HRV
### processing values: next_to_last_VIS006 last_image_VIS006
### processing values: next_to_last_VIS008 last_image_VIS008
### processing values: next_to_last_IR_016 last_image_IR_016
### processing values: next_to_last_IR_039 last_image_IR_039
### processing values: next_to_last_WV_062 last_image_WV_062
### processing values: next_to_last_WV_073 last_image_WV_073
### processing values: next_to_last_IR_087 last_image_IR_087
### processing values: next_to_last_IR_097 last_image_IR_097
### processing values: next_to_last_IR_108 last_image_IR_108
### processing values: next_to_last_IR_120 last_image_IR_120
### processing values: next_to_last_IR_134 last_image_IR_134

### process

In [21]:
# Function: regression_results to calculate and print regression metrics
def regression_results(y_true, y_pred):

    results = {}  # create empty dictionary to store results

    # set regression metrics
    explained_variance=metrics.explained_variance_score(y_true, y_pred)
    mean_absolute_error=metrics.mean_absolute_error(y_true, y_pred) 
    mse=metrics.mean_squared_error(y_true, y_pred) 
    #mean_squared_log_error=metrics.mean_squared_log_error(y_true, y_pred)
    median_absolute_error=metrics.median_absolute_error(y_true, y_pred)
    
    # Calculate R2 manually to avoid error with only one sample
    #r2=metrics.r2_score(y_true, y_pred)
    results = {}  # Create empty dictionary to store results
    if len(y_true) < 2:
        results['r2'] = np.nan
        print("R² skipped: <2 Samples")
    else:
        results['r2'] = r2_score(y_true, y_pred)
        print('R²:', round(results['r2'], 4))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    print('explained_variance: ', round(explained_variance,4))
    # print('mean_squared_log_error: ', round(mean_squared_log_error,4))
    print('MAE: ', round(mean_absolute_error,4))
    print('MSE: ', round(mse,4))
    print('RMSE: ', round(np.sqrt(mse),4))
    print("Dataframe SHAPE:",y_true.shape, y_pred.shape)
    print('MAPE: ',round(mape,4))
    # print(y_true.head(5), y_pred.head(5))
    # print(y_true.describe())
    # print(y_pred.describe())

    y_true_reset = y_true.reset_index(drop=True)
    y_pred_reset = y_pred.reset_index(drop=True)
    forecast_errors = [y_true_reset[i] - y_pred_reset[i] for i in range(len(y_true_reset))]
    bias = sum(forecast_errors) * 1.0 / len(y_true_reset)
    print('Bias: %f' % bias)

    # Mean Arctangent Absolute Percentage Error (MAAPE)
    # Source: https://gist.github.com/bshishov/5dc237f59f019b26145648e2124ca1c9
    EPSILON = 1e-10
    maape = np.mean(np.arctan(np.abs((y_true - y_pred) / (y_pred + EPSILON))))
    print('MAAPE: ',round(maape,4))

    # fill dictonary with metrics
    results['explained Variance'] = explained_variance
    results['MAE'] = mean_absolute_error
    results['MSE'] = mse
    results['RMSE'] = np.sqrt(mse)
    results['MAPE'] = mape
    results['Bias'] = bias
    results['MAAPE'] = maape
    results['N Samples'] = len(y_true)

    return results

loop over all Feature-Sets and matching CBM-models for CatBoost prediction

In [ ]:
# Target variable
TARGET = 'SPECIFIC_YIELD'

# path to the folder with all trained cbm files
model_path = './Models'

# dataset selection
csv_dataset = "alle_151_Netzeinspeiser"
#csv_dataset = "Radius_5000m"
#csv_dataset = "Radius_200m"

dataframe = pd.read_csv('../../Daten/Solar_Load_Profile/2018_Features_Solar_Load_Profile_' + csv_dataset + '.csv',parse_dates=['TIMESTAMP'])

# time-based cyclic features
dataframe['HOURDEZ'] = (pd.to_datetime(dataframe['TIMESTAMP']).dt.hour + pd.to_datetime(dataframe['TIMESTAMP']).dt.minute / 60)
dataframe['DAYYEAR'] = pd.to_datetime(dataframe['TIMESTAMP']).dt.dayofyear
dataframe['SIN_HOUR'] = np.sin(2 * np.pi * dataframe['HOURDEZ'] / 24)
dataframe['COS_HOUR'] = np.cos(2 * np.pi * dataframe['HOURDEZ'] / 24)
dataframe['SIN_DAY'] = np.sin(2 * np.pi * dataframe['DAYYEAR'] / 365)
dataframe['COS_DAY'] = np.cos(2 * np.pi * dataframe['DAYYEAR'] / 365)

# sort like in training notebook
dataframe.sort_values(by=['TIMESTAMP', 'NEI_ID'], inplace=True)

print(dataframe.shape)

(2269939, 40)


main loop: scenario x feature-set x leadtime

In [15]:
# empty list to collect result-rows
all_results = []

# separate list for the aggregated metrics per scenario/feature-set combo
metrics_results = []

# loop through all scenarios that were already forecasted with optical flow above
for scenario_result in all_scenario_results:

    scenario_name = scenario_result["scenario"]
    date_predict = scenario_result["timestamp"]
    forecast_results = scenario_result["forecast_results"]

    # get rounded timestamps again (same helper function as above in the notebook)
    date_last_image_rounded, date_last_image, date_next_to_last_image = compute_rounded_times(date_predict)

    print(f"\n#### Scenario {scenario_name} ({date_predict})")

    # loop trough all feature sets - model already loaded, no need to load per scenario anymore
    for fs_name, fs_conf in FEATURE_SETS.items():

        # get feature-list and the matching preloaded model from the dictionary built above
        fs_columns = fs_conf['features']
        model = loaded_models[fs_name]

        # check: catches stale/duplicate FEATURE_SETS
        if not isinstance(fs_columns, list):
            raise TypeError(f"FEATURE_SETS['{fs_name}']['features'] is a {type(fs_columns)}, expected a list!")

        print("### using model:", fs_name, "-", fs_conf['cbm'])

        # counter for the 15min forecast steps, same logic as original script
        counter = 0

        # dataframe just to sum things up per feature-set/scenario for the plots later
        dataframe_result = pd.DataFrame()

        # count skipped steps to warn if a whole scenario/feature-set combo has zero usable data
        skipped_steps = 0

        while counter <= forecast_range:

            forecast_name = date_last_image + " +" + str(counter) + "min"
            date_prediction = date_last_image_rounded + datetime.timedelta(minutes=counter)

            # filter dataframe for the current prediction timestamp
            timefiltered_dataframe = dataframe[dataframe['TIMESTAMP'] == date_prediction].copy()

            # skip if no rows found (missing csv rows for that timestamp)
            if timefiltered_dataframe.empty:
                print("no rows for", date_prediction, "-> skip")
                skipped_steps += 1
                counter += forecast_step
                continue

            # get coordinates for sampling the forecast geotiff
            coords = [(x, y) for x, y in zip(timefiltered_dataframe.LON_WGS84, timefiltered_dataframe.LAT_WGS84)]

            # only loop over the bands that are actually needed for this feature set
            bands_needed = [b for b in all_bands if b in fs_columns]

            for band in bands_needed:
                forecast_band_name = forecast_name + "_" + band
                forecast_file = os.path.join(eumetsat_forecast_output_path + "\\" + band, forecast_band_name + ".tif")

                # not every band-tif might exist for every leadtime, so wrap in try/except
                try:
                    forecast_raster = rasterio.open(forecast_file)
                except Exception as e:
                    print("could not open File! ", forecast_file, e)
                    continue

                # rename original column and overwrite with new pixel-sampled value
                if band in timefiltered_dataframe.columns:
                    timefiltered_dataframe.rename(columns={band: band + "_ORG"}, inplace=True)

                timefiltered_dataframe[band] = [x[0] for x in forecast_raster.sample(coords)]

            # select only the columns needed for this feature set - list() cast as extra safety net
            features = timefiltered_dataframe[list(fs_columns)].copy()

            # predict SPECIFIC_YIELD with the preloaded model for this feature-set
            y_pred = model.predict(features)

            timefiltered_dataframe['SPECIFIC_YIELD_PREDICTION'] = y_pred
            timefiltered_dataframe['SPECIFIC_YIELD_ERROR'] = timefiltered_dataframe['SPECIFIC_YIELD_PREDICTION'] - timefiltered_dataframe['SPECIFIC_YIELD']

            # tag rows with scenario/feature-set/leadtime info -> needed later for grouping/plots
            timefiltered_dataframe['SCENARIO'] = scenario_name
            timefiltered_dataframe['FEATURE_SET'] = fs_name
            timefiltered_dataframe['LEADTIME_MIN'] = counter

            dataframe_result = pd.concat([dataframe_result, timefiltered_dataframe], ignore_index=True)

            # NEW: also compute + collect per-leadtime metrics (across all PV feeders at this timestamp)
            # this gives us the RMSE-vs-leadtime curve later without re-grouping raw rows
            step_metrics = regression_results(timefiltered_dataframe["SPECIFIC_YIELD"], timefiltered_dataframe["SPECIFIC_YIELD_PREDICTION"])
            step_metrics["SCENARIO"] = scenario_name
            step_metrics["FEATURE_SET"] = fs_name
            step_metrics["LEADTIME_MIN"] = counter
            step_metrics["TIMESTAMP"] = date_prediction
            metrics_results.append(step_metrics)

            counter += forecast_step

        # warn explicitly if a scenario/feature-set combo produced zero usable rows
        # (this is what caused the "only 4 of 6 scenarios" issue before - silent skip is dangerous)
        total_steps = (forecast_range // forecast_step) + 1
        if skipped_steps == total_steps:
            print(f"!!! WARNING: '{scenario_name}' / {fs_name} has ZERO usable timesteps - check CSV for {date_predict.date()}")

        # append this feature-set's results (all leadtimes) to the big collector list
        if not dataframe_result.empty:
            all_results.append(dataframe_result)

            # print quick regression check per feature-set/scenario, grouped by timestamp
            grouped_check = dataframe_result[["TIMESTAMP", "SPECIFIC_YIELD", "SPECIFIC_YIELD_PREDICTION"]].groupby("TIMESTAMP").sum().reset_index()
            print(f"--- {scenario_name} / {fs_name} (overall) ---")
            overall_metrics = regression_results(grouped_check["SPECIFIC_YIELD"], grouped_check["SPECIFIC_YIELD_PREDICTION"])

            # NEW: also store the "overall" per-scenario/feature-set metric as a separate row
            # (LEADTIME_MIN = "ALL" marks this as the aggregated row, not a single leadtime)
            overall_metrics["SCENARIO"] = scenario_name
            overall_metrics["FEATURE_SET"] = fs_name
            overall_metrics["LEADTIME_MIN"] = "ALL"
            overall_metrics["TIMESTAMP"] = pd.NaT
            metrics_results.append(overall_metrics)

# combine everything into one dataframe for plotting/metrics
df_all = pd.concat(all_results, ignore_index=True)
print(df_all.shape)
df_all.head()

df_metrics = pd.DataFrame(metrics_results)

# reorder columns
id_cols = ["SCENARIO", "FEATURE_SET", "LEADTIME_MIN", "TIMESTAMP"]
metric_cols = [c for c in df_metrics.columns if c not in id_cols]
df_metrics = df_metrics[id_cols + metric_cols]

print(df_metrics.shape)
df_metrics.head(15)

# save both dataframes to disk - raw predictions + aggregated metrics
df_all.to_csv(os.path.join(result_path, "prediction_results_raw.csv"), index=False,sep=";",decimal=",",encoding="utf-8-sig")
df_metrics.to_csv(os.path.join(result_path, "prediction_metrics_summary.csv"), index=False,sep=";",decimal=",",encoding="utf-8-sig")

actual/predict datetime:             2018-01-29 07:35:11
rounded datetime last image:         2018-01-29 07_30_00
rounded datetime next to last image: 2018-01-29 07_15_00

#### Scenario Clear winter morning with shadows from the mountains (2018-01-29 07:35:11)
### using model: FS1_BASELINE - ./Models/model_FS1_BASELINE.cbm
R²: 0.186
explained_variance:  0.2903
MAE:  0.0257
MSE:  0.0017
RMSE:  0.0413
Dataframe SHAPE: (132,) (132,)
MAPE:  4.0416
Bias: -0.014793
MAAPE:  0.6111
R²: 0.1942
explained_variance:  0.1996
MAE:  0.0261
MSE:  0.0031
RMSE:  0.0555
Dataframe SHAPE: (133,) (133,)
MAPE:  1.4029
Bias: 0.004546
MAAPE:  0.5071
R²: 0.072
explained_variance:  0.1731
MAE:  0.0425
MSE:  0.0057
RMSE:  0.0758
Dataframe SHAPE: (134,) (134,)
MAPE:  1.2271
Bias: 0.025014
MAAPE:  0.6335
R²: -0.1228
explained_variance:  0.1636
MAE:  0.0624
MSE:  0.0096
RMSE:  0.0978
Dataframe SHAPE: (134,) (134,)
MAPE:  0.9254
Bias: 0.049375
MAAPE:  0.7513
R²: -0.3788
explained_variance:  0.1631
MAE:  0.0872
MSE:  

Main predict loop: scenario x feature-set x leadtime

In [ ]:
# create empty list to collect raw prediction rows (all PV feeders, all leadtimes)
all_results = []

# create separate list just for the aggregated metrics per scenario/feature-set combo
metrics_results = []

# loop through all scenarios that were already forecasted with optical flow above
for scenario_result in all_scenario_results:

    scenario_name = scenario_result["scenario"]
    date_predict = scenario_result["timestamp"]
    forecast_results = scenario_result["forecast_results"]

    # get rounded timestamps
    date_last_image_rounded, date_last_image, date_next_to_last_image = compute_rounded_times(date_predict)

    print(f"\nScenario {scenario_name} ({date_predict})")

    # loop trough all feature sets - model already loaded, no need to load per scenario anymore
    for fs_name, fs_conf in FEATURE_SETS.items():

        # get feature-list and the matching preloaded model from the dictionary built above
        fs_columns = fs_conf['features']
        model = loaded_models[fs_name]

        # check: catches stale/duplicate FEATURE_SETS
        if not isinstance(fs_columns, list):
            raise TypeError(f"FEATURE_SETS['{fs_name}']['features'] is a {type(fs_columns)}, expected a list!")

        print("using model:", fs_name, "-", fs_conf['cbm'])

        # counter for the 15min forecast steps
        counter = 0

        # dataframe just to sum things up
        dataframe_result = pd.DataFrame()

        # count skipped steps by zero rows for a given timestamp
        skipped_steps = 0

        while counter <= forecast_range:

            forecast_name = date_last_image + " +" + str(counter) + "min"
            date_prediction = date_last_image_rounded + datetime.timedelta(minutes=counter)

            # filter dataframe for the current prediction timestamp
            timefiltered_dataframe = dataframe[dataframe['TIMESTAMP'] == date_prediction].copy()

            # skip if no rows found (missing csv rows for that timestamp)
            if timefiltered_dataframe.empty:
                print("no rows for", date_prediction, "-> skip")
                skipped_steps += 1
                counter += forecast_step
                continue

            # get coordinates for sampling the forecast geotiff
            coords = [(x, y) for x, y in zip(timefiltered_dataframe.LON_WGS84, timefiltered_dataframe.LAT_WGS84)]

            # only loop over the bands that are actually needed for this feature set
            bands_needed = [b for b in all_bands if b in fs_columns]

            for band in bands_needed:
                forecast_band_name = forecast_name + "_" + band
                forecast_file = os.path.join(eumetsat_forecast_output_path + "\\" + band, forecast_band_name + ".tif")

                # not every band-tif might exist for every leadtime, so wrap in try/except
                try:
                    forecast_raster = rasterio.open(forecast_file)
                except Exception as e:
                    print("could not open File! ", forecast_file, e)
                    continue

                # rename original column and overwrite with new pixel-sampled value
                if band in timefiltered_dataframe.columns:
                    timefiltered_dataframe.rename(columns={band: band + "_ORG"}, inplace=True)

                timefiltered_dataframe[band] = [x[0] for x in forecast_raster.sample(coords)]

            # select only the columns needed for this feature set
            features = timefiltered_dataframe[list(fs_columns)].copy()

            # predict SPECIFIC_YIELD with the preloaded model for this feature-set
            y_pred = model.predict(features)

            timefiltered_dataframe['SPECIFIC_YIELD_PREDICTION'] = y_pred
            timefiltered_dataframe['SPECIFIC_YIELD_ERROR'] = timefiltered_dataframe['SPECIFIC_YIELD_PREDICTION'] - timefiltered_dataframe['SPECIFIC_YIELD']

            # tag rows with scenario/feature-set/leadtime info
            timefiltered_dataframe['SCENARIO'] = scenario_name
            timefiltered_dataframe['FEATURE_SET'] = fs_name
            timefiltered_dataframe['LEADTIME_MIN'] = counter

            dataframe_result = pd.concat([dataframe_result, timefiltered_dataframe], ignore_index=True)

            # compute + collect per-leadtime metrics (across all PV feeders at this timestamp)
            step_metrics = regression_results(timefiltered_dataframe["SPECIFIC_YIELD"], timefiltered_dataframe["SPECIFIC_YIELD_PREDICTION"])
            step_metrics["SCENARIO"] = scenario_name
            step_metrics["FEATURE_SET"] = fs_name
            step_metrics["LEADTIME_MIN"] = counter
            step_metrics["TIMESTAMP"] = date_prediction
            metrics_results.append(step_metrics)

            counter += forecast_step

        # append this feature-set's results (all leadtimes) to the big collector list
        if not dataframe_result.empty:
            all_results.append(dataframe_result)

            # OPTION A: pool ALL individual rows (all plants x all leadtimes) instead of summing per timestamp
            # this keeps n_samples in the hundreds range (e.g. ~130 plants x 13 leadtimes) - way more robust
            # than the old grouped-sum approach which collapsed everything down to just 13 rows
            print(f"--- {scenario_name} / {fs_name} (overall, pooled) ---")
            overall_metrics = regression_results(dataframe_result["SPECIFIC_YIELD"], dataframe_result["SPECIFIC_YIELD_PREDICTION"])

            # store the "overall" as LEADTIME_MIN = "ALL" per-scenario/feature-set metric as a separate row
            overall_metrics["SCENARIO"] = scenario_name
            overall_metrics["FEATURE_SET"] = fs_name
            overall_metrics["LEADTIME_MIN"] = "ALL"
            overall_metrics["TIMESTAMP"] = pd.NaT
            metrics_results.append(overall_metrics)

# combine everything into one big dataframe for plotting/metrics
df_all = pd.concat(all_results, ignore_index=True)
print(df_all.shape)
df_all.head()

# export structured metrics as CSV
df_metrics = pd.DataFrame(metrics_results)

# reorder columns so the ID-columns come first, easier to read in Excel
id_cols = ["SCENARIO", "FEATURE_SET", "LEADTIME_MIN", "TIMESTAMP"]
metric_cols = [c for c in df_metrics.columns if c not in id_cols]
df_metrics = df_metrics[id_cols + metric_cols]

print(df_metrics.shape)
df_metrics.head(15)

# save both dataframes to disk - raw predictions + aggregated metrics
df_all.to_csv(os.path.join(result_path, "prediction_results_raw.csv"), index=False,sep=";",decimal=",",encoding="utf-8-sig")
df_metrics.to_csv(os.path.join(result_path, "prediction_metrics_summary.csv"), index=False,sep=";",decimal=",",encoding="utf-8-sig")

actual/predict datetime:             2018-01-29 07:35:11
rounded datetime last image:         2018-01-29 07_30_00
rounded datetime next to last image: 2018-01-29 07_15_00

#### Scenario Clear winter morning with shadows from the mountains (2018-01-29 07:35:11)
### using model: FS1_BASELINE - ./Models/model_FS1_BASELINE.cbm
R²: 0.186
explained_variance:  0.2903
MAE:  0.0257
MSE:  0.0017
RMSE:  0.0413
Dataframe SHAPE: (132,) (132,)
MAPE:  4.0416
Bias: -0.014793
MAAPE:  0.6111
R²: 0.1942
explained_variance:  0.1996
MAE:  0.0261
MSE:  0.0031
RMSE:  0.0555
Dataframe SHAPE: (133,) (133,)
MAPE:  1.4029
Bias: 0.004546
MAAPE:  0.5071
R²: 0.072
explained_variance:  0.1731
MAE:  0.0425
MSE:  0.0057
RMSE:  0.0758
Dataframe SHAPE: (134,) (134,)
MAPE:  1.2271
Bias: 0.025014
MAAPE:  0.6335
R²: -0.1228
explained_variance:  0.1636
MAE:  0.0624
MSE:  0.0096
RMSE:  0.0978
Dataframe SHAPE: (134,) (134,)
MAPE:  0.9254
Bias: 0.049375
MAAPE:  0.7513
R²: -0.3788
explained_variance:  0.1631
MAE:  0.0872
MSE:  

### Diagrams

In [17]:
# group per scenario / feature-set / leadtime -> sum over all PV feeders
df_grouped = df_all.groupby(["SCENARIO", "FEATURE_SET", "LEADTIME_MIN"])[["SPECIFIC_YIELD", "SPECIFIC_YIELD_PREDICTION"]].sum().reset_index()

fig = px.line(df_grouped, x="LEADTIME_MIN", y="SPECIFIC_YIELD_PREDICTION", color="FEATURE_SET",
               facet_col="SCENARIO", facet_col_wrap=2,
               title="Predicted Specific Yield per Feature-Set and Leadtime (all Scenarios)")

# observed/true values as a dashed black reference line per facet
# loop trough scenarios because px does not support mixing two y-columns easily)
scenarios = df_grouped["SCENARIO"].unique()
for i, scen in enumerate(scenarios):
    obs = df_grouped[(df_grouped["SCENARIO"] == scen) & (df_grouped["FEATURE_SET"] == df_grouped["FEATURE_SET"].iloc[0])]
    fig.add_scatter(x=obs["LEADTIME_MIN"], y=obs["SPECIFIC_YIELD"], mode="lines",
                     name="Observed" if i == 0 else None, showlegend=(i == 0),
                     line=dict(color="black", dash="dash"),
                     row=(i // 2) + 1, col=(i % 2) + 1)

fig.update_yaxes(title_text="Specific Yield")
fig.update_xaxes(title_text="Leadtime (min)")
fig.show()
fig.write_html(result_path + "/prediction_timeseries_all_scenarios.html")  # keep interactive version as result

# Diagram: for research question TF2 - error vs leadtime per feature-set

def calc_rmse(group):
    return np.sqrt(np.mean((group["SPECIFIC_YIELD"] - group["SPECIFIC_YIELD_PREDICTION"])**2))

rmse_per_leadtime = df_all.groupby(["FEATURE_SET", "LEADTIME_MIN"]).apply(calc_rmse).reset_index(name="RMSE")

fig2 = px.line(rmse_per_leadtime, x="LEADTIME_MIN", y="RMSE", color="FEATURE_SET",
                title="RMSE of Specific Yield Prediction over Forecast Leadtime (TF2)",
                markers=True)
fig2.update_xaxes(title_text="Leadtime (min)")
fig2.update_yaxes(title_text="RMSE")
fig2.show()
fig2.write_html(result_path + "/prediction_timeseries_error_vs_leadtime_per_feature-set.html")  # keep interactive version as result

# %% [markdown]
# ### NEW: Diagram for research question TF4 - feature-set comparison (boxplot of errors)
# quick overview which feature-set has the lowest spread of errors across all scenarios

# %%
fig3 = px.box(df_all, x="FEATURE_SET", y="SPECIFIC_YIELD_ERROR", color="FEATURE_SET",
               title="Distribution of Prediction Error per Feature-Set (TF4)")
fig3.update_xaxes(title_text="Feature-Set")
fig3.update_yaxes(title_text="Error (pred - true)")
fig3.show()
fig3.write_html(result_path + "/prediction_timeseries_feature-set_comparison.html")  # keep interactive version as result

# save the big result dataframe for further analysis / thesis appendix
df_all.to_csv(result_path + "/prediction_results_all_scenarios_featuresets.csv", index=False)


In [18]:
# %% [markdown]
# ### NEW: plot forecast curves - measured (Ist) vs all feature-sets, aggregated per scenario
# aggregation: mean over all PV-plants per timestamp (sum would overweight big plants)

# %%
# build one aggregated dataframe: mean SPECIFIC_YIELD / PREDICTION per scenario x featureset x leadtime
plot_df = df_all.groupby(["SCENARIO", "FEATURE_SET", "LEADTIME_MIN"]).agg(
    SPECIFIC_YIELD=("SPECIFIC_YIELD", "mean"),
    SPECIFIC_YIELD_PREDICTION=("SPECIFIC_YIELD_PREDICTION", "mean")
).reset_index()

# melt predictions into long-format so every feature-set becomes its own colored line
pred_long = plot_df[["SCENARIO", "FEATURE_SET", "LEADTIME_MIN", "SPECIFIC_YIELD_PREDICTION"]].copy()
pred_long.rename(columns={"SPECIFIC_YIELD_PREDICTION": "VALUE"}, inplace=True)
pred_long["SERIES"] = pred_long["FEATURE_SET"]  # color-key = feature-set name

# measured values are identical for every feature-set at a given scenario/leadtime -> just take FS1 rows
ist_long = plot_df[plot_df["FEATURE_SET"] == "FS1_BASELINE"][["SCENARIO", "LEADTIME_MIN", "SPECIFIC_YIELD"]].copy()
ist_long.rename(columns={"SPECIFIC_YIELD": "VALUE"}, inplace=True)
ist_long["SERIES"] = "IST (measured)"

# stack together - Ist as separate "series" so it gets its own line/color
plot_long = pd.concat([pred_long[["SCENARIO", "LEADTIME_MIN", "SERIES", "VALUE"]],
                        ist_long[["SCENARIO", "LEADTIME_MIN", "SERIES", "VALUE"]]], ignore_index=True)

fig1 = px.line(plot_long, x="LEADTIME_MIN", y="VALUE", color="SERIES",
                facet_col="SCENARIO", facet_col_wrap=2,
                title="SPECIFIC_YIELD forecast vs measured - all feature-sets per scenario",
                labels={"LEADTIME_MIN": "Leadtime [min]", "VALUE": "SPECIFIC_YIELD [kWh/kWp]"})

# make the "Ist" line stand out - black, dashed, thicker line width
for trace in fig1.data:
    if trace.name == "IST (measured)":
        trace.line.color = "black"
        trace.line.dash = "dash"
        trace.line.width = 3

fig1.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5))
#fig1.write_image(os.path.join(result_path, "forecast_vs_measured_allFS.png"))
fig1.show()
fig1.write_html(result_path + "/forecast_vs_measured_allFS.html")  # keep interactive version as result

Heatmaps R2, RMSE, MAAPE and BIAS - for each Scenario and Feature-Set

In [38]:
metrics = ["r2", "RMSE", "MAAPE", "Bias"]

# Filter only the aggregated "ALL" rows from array
all_df = df_metrics[df_metrics["LEADTIME_MIN"] == "ALL"].copy()

for metric in metrics:
    # 1. Pivot into matrix form for the heatmap
    pivot_matrix = all_df.pivot(index="SCENARIO", columns="FEATURE_SET", values=metric)
    print(f"Shape for {metric.upper()}: {pivot_matrix.shape}")

    # Use full feature-set names (no shortening)
    x_labels = [str(col) for col in pivot_matrix.columns]
    y_labels = [str(idx) for idx in pivot_matrix.index]

    # 2. Handle clipping / styling depending on the metric
    if metric == "r2":
        z_clipped = pivot_matrix.clip(-2, 1).values
        z_text = pivot_matrix.round(2).astype(str).values
        colorscale = "RdYlGn"
        zmid, zmin, zmax = 0, -2, 1
        colorbar_title = "R2"
        title_text = "Heatmap für Metrik R² je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = [-2, -1, 0, 1]
    elif metric == "RMSE":
        # lower RMSE is better -> no hard clipping needed, adjust if outliers dominate the scale
        z_clipped = pivot_matrix.values
        z_text = pivot_matrix.round(2).astype(str).values
        colorscale = "RdYlGn_r"  # _r inverts scale because smaller RMSE = better
        zmid, zmin, zmax = None, None, None
        colorbar_title = "RMSE"
        title_text = "Heatmap für Metrik RMSE je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = None
    elif metric == "MAAPE":
        # smaller MAAPE is better
        z_clipped = pivot_matrix.values
        z_text = pivot_matrix.round(2).astype(str).values
        colorscale = "RdYlGn_r"
        zmid, zmin, zmax = None, None, None
        colorbar_title = "MAAPE"
        title_text = "Heatmap für Metrik MAAPE je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = None
    else:  # bias
        # bias close to 0 is good - positive = model underestimates, negative = model overestimates
        # symmetric clipping around 0, use max absolute value across the matrix (ignore extreme scenario outliers)
        bias_limit = min(pivot_matrix.abs().quantile(0.9).max(), 0.5)  # cap so a single crazy scenario doesn't flatten colors
        z_clipped = pivot_matrix.clip(-bias_limit, bias_limit).values
        z_text = pivot_matrix.round(3).astype(str).values  # more decimals, bias values are usually small
        colorscale = "RdBu"  # diverging: blue = underestimate, red = overestimate
        zmid, zmin, zmax = 0, -bias_limit, bias_limit
        colorbar_title = "Bias"
        title_text = "Heatmap für Metrik Bias je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = None

    # 3. Create Figure
    fig = go.Figure(data=go.Heatmap(
        z=z_clipped,
        x=x_labels,
        y=y_labels,
        text=z_text,
        texttemplate="%{text}",
        textfont=dict(size=12),
        colorscale=colorscale,
        zmid=zmid,
        zmin=zmin,
        zmax=zmax,
        colorbar=dict(title=colorbar_title, tickvals=tickvals)
    ))

    fig.update_layout(
        title=title_text,
        height=650,
        width=1200,  # a bit wider because of the full feature-set names
        xaxis=dict(tickangle=-45)  # tilt long names for better readability
    )
    fig.update_xaxes(title_text="Feature-Set")
    fig.update_yaxes(title_text="Szenario", automargin=True)

    # 4. Save and show
    file_name = f"/{metric.lower()}_heatmap_scenario_featureset.png"
    fig.write_image(result_path + file_name, scale=3)
    fig.show()

Shape for R2: (7, 6)


Shape for RMSE: (7, 6)


Shape for MAAPE: (7, 6)


Shape for BIAS: (7, 6)
